# Домашняя работа 3

## Подготовка

In [4]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## Исходники домашней работы

In [5]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [6]:
work = decisions.copy()
work["is_denied"] = work["action_taken"].eq(3)

denial = work.groupby(org_keys, as_index=False).agg(
    n_decisions=("is_denied", "size"),
    n_denied=("is_denied", "sum"),
)
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]

profile = denial.merge(
    lookup, on=org_keys, how="left", validate="one_to_one", indicator=True
)

# покрытие справочника
print(profile["_merge"].value_counts())
print("Групп без имени:", profile["organization_name"].isna().sum())

# проверки по исходным данным
assert len(profile) == len(denial)
assert profile["n_decisions"].sum() == len(decisions)
assert profile["n_denied"].sum() == work["is_denied"].sum()
assert profile["denial_rate"].between(0, 1).all()

profile.to_csv(WORK / "home_denial_profile.csv", index=False)

# одна организация: числитель и знаменатель
row = profile.iloc[0]
print(row["lei"], row["activity_year"], row["organization_name"])
print("Числитель:", row["n_denied"], "| Знаменатель:", row["n_decisions"],
      f"| Доля: {100 * row['denial_rate']:.2f}%")

_merge
both          314
left_only       0
right_only      0
Name: count, dtype: int64
Групп без имени: 0
03D0JEWFDFUS0SEEKG89 2023 TD Bank, National Association
Числитель: 15 | Знаменатель: 91 | Доля: 16.48%


## H2 Группы для просмотра

In [7]:
groups_before = len(denial)
large_groups = denial[denial["n_decisions"].ge(20)].copy()
print("Групп было:", groups_before, "| осталось:", len(large_groups))
assert len(denial) == groups_before   # полный denial сохранён

large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)

overall_rate = denial["n_denied"].sum() / denial["n_decisions"].sum()
large_rate = large_groups["n_denied"].sum() / large_groups["n_decisions"].sum()
mean_of_rates = denial["denial_rate"].mean()

print(f"Общая доля (сумма отказов / сумма решений): {100 * overall_rate:.2f}%")
print(f"Доля в группах от 20 решений:               {100 * large_rate:.2f}%")
print(f"Простое среднее долей групп:                {100 * mean_of_rates:.2f}%")
print("Решений в больших группах:", large_groups["n_decisions"].sum(),
      "из", denial["n_decisions"].sum())

Групп было: 314 | осталось: 61
Общая доля (сумма отказов / сумма решений): 10.12%
Доля в группах от 20 решений:               8.88%
Простое среднее долей групп:                14.22%
Решений в больших группах: 5768 из 6830


## H3 Длинная, широкая и снова длинная форма

In [8]:
long_counts = (
    decisions.groupby(long_keys, as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

wide = (
    long_counts.pivot(index=org_keys, columns="action_taken", values="n")
    .reindex(columns=[1, 2, 3])
    .fillna(0)
    .astype("int64")
)
print("Нулей (отсутствующих сочетаний):", int((wide == 0).sum().sum()))

back = wide.reset_index().melt(
    id_vars=org_keys, var_name="action_taken", value_name="n"
)
back["action_taken"] = back["action_taken"].astype("int64")
back["n"] = back["n"].astype("int64")
back = back[back["n"] > 0]

left = long_counts.sort_values(long_keys).reset_index(drop=True)
right = back.sort_values(long_keys).reset_index(drop=True)
assert_frame_equal(left, right)

assert wide.to_numpy().sum() == len(decisions)
assert right["n"].sum() == len(decisions)
print("Форма восстановлена:", right.shape)

Нулей (отсутствующих сочетаний): 442
Форма восстановлена: (500, 4)


## H+ Безопасное соединение

In [9]:
lookup_full = lookup.copy()
lookup_partial = lookup.iloc[: len(lookup) // 2].copy()
lookup_repeated = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)

conflict_row = lookup.iloc[[0]].copy()
conflict_row["organization_name"] = "Конфликтное имя"
lookup_conflict = pd.concat([lookup, conflict_row], ignore_index=True)

names_per_key = lookup_conflict.groupby(org_keys)["organization_name"].nunique(dropna=False)
print("Максимум различных имён у одного ключа:", names_per_key.max())


def safe_enrich(left, right, keys, value="organization_name"):
    missing_left = [k for k in keys if k not in left.columns]
    missing_right = [k for k in keys + [value] if k not in right.columns]
    if missing_left or missing_right:
        raise KeyError(f"Неизвестные ключи: слева {missing_left}, справа {missing_right}")
    if right.duplicated(keys).any():
        raise ValueError("Повторный ключ справа: соединение остановлено")

    merged = left.merge(
        right[keys + [value]], on=keys, how="left",
        validate="many_to_one", indicator=True,
    )
    assert len(merged) == len(left)
    left_only = int((merged["_merge"] == "left_only").sum())
    enriched = merged.drop(columns="_merge")
    assert_frame_equal(enriched[left.columns], left.reset_index(drop=True))
    report = {"left_rows": len(left), "left_only": left_only}
    return enriched, report


copies = {
    "full": lookup_full,
    "partial": lookup_partial,
    "repeated": lookup_repeated,
    "conflict": lookup_conflict,
}
results = {}
for name, right in copies.items():
    try:
        _, report = safe_enrich(denial, right, org_keys)
        results[name] = ("ok", report)
    except (KeyError, ValueError) as err:
        results[name] = ("ошибка", str(err))
    print(name, results[name])

assert results["full"][0] == "ok" and results["partial"][0] == "ok"
assert results["repeated"][0] == "ошибка" and results["conflict"][0] == "ошибка"
assert results["partial"][1]["left_only"] > 0

try:
    safe_enrich(denial, lookup, ["lei", "no_such_key"])
except KeyError as err:
    print("Ожидаемая ошибка:", err)

Максимум различных имён у одного ключа: 2
full ('ok', {'left_rows': 314, 'left_only': 0})
partial ('ok', {'left_rows': 314, 'left_only': 167})
repeated ('ошибка', 'Повторный ключ справа: соединение остановлено')
conflict ('ошибка', 'Повторный ключ справа: соединение остановлено')
Ожидаемая ошибка: "Неизвестные ключи: слева ['no_such_key'], справа ['no_such_key']"
